# CN7 가스: 정상 대비 충전·냉각 동시 변화
170번 이후 놓친6개와 오탐을 검토한 개발 실험. 시간/Run ID는 예측 변수로 사용하지 않는다.
작업 전 .venv 활성화. 170번과 외부 163/165 입력 파일이 필요하다.


In [1]:
from pathlib import Path
import nbformat
# Reuse audited split and training-only relation fits without rerunning notebook170.
old=nbformat.read("notebooks/170_cn7_gas_persistent_relation_experts.ipynb",as_version=4)
oldcode="\n".join(c.source for c in old.cells if c.cell_type=="code")
exec(oldcode[:oldcode.index("def predict(parent,q):")])
VIEW_NAMES=["flow_signed6","thermal_flow_joint4","meter_load6","flow_history8"]
HEADS=["LR","RF","ET","mean3"]
BASIC=["Mold_Temperature_3","Mold_Temperature_4","Max_Injection_Speed","Filling_Time",
 "Max_Injection_Pressure","Max_Switch_Over_Pressure","Plasticizing_Position",
 "Plasticizing_Time","Cushion_Position","Average_Back_Pressure","Max_Screw_RPM"]
raw=d[[v+"_raw" for v in BASIC]].to_numpy(float)
def normal_scores(ref):
 normal=ref[y[ref]==0]
 a=raw.copy();out=np.empty_like(a)
 for run in pd.unique(g):
  ni=normal[g[normal]==run]
  if len(ni)<8:ni=normal
  imp=SimpleImputer(keep_empty_features=True).fit(a[ni])
  aa=imp.transform(a)
  # std floor uses sensor precision learned from reference normal only.
  floor=[]
  for k in range(aa.shape[1]):
   vals=np.unique(np.round(aa[ni,k],4));dif=np.diff(vals);nz=dif[dif>1e-8]
   floor.append(max(float(np.min(nz)) if len(nz) else .01,1e-4))
  scale=np.maximum(np.std(aa[ni],axis=0),np.array(floor))
  ids=np.flatnonzero(g==run)
  out[ids]=(aa[ids]-np.median(aa[ni],axis=0))/scale
 return np.clip(out,-20,20)
scorememo={}
def views(z,ctx,ids,ref):
 key=tuple(ref)
 if key not in scorememo:scorememo[key]=normal_scores(ref)
 ns=scorememo[key]
 mold3,mold4,speed,fill,pressure,switch,meter,plastic,cushion,bp,rpm=ns.T
 flow=np.column_stack([mold3,speed,fill,pressure,switch,pressure-speed])
 # Low mold temperature, faster filling, reduced pressure: hypotheses, not proven causes.
 joint=np.column_stack([np.minimum(-mold3,speed),np.minimum(-fill,speed),
  np.minimum(-pressure,speed),np.minimum(-mold3,-fill)])
 load=np.column_stack([meter,plastic,cushion,bp,rpm,meter-cushion])
 frame=pd.DataFrame(np.column_stack([speed,fill,pressure,mold3]),index=np.arange(N))
 ii=np.flatnonzero(ctx)
 segment=np.cumsum(np.r_[True,(g[ii][1:]!=g[ii][:-1])|(np.diff(ii)!=1)])
 hist=frame.iloc[ii].groupby(segment,group_keys=False).apply(lambda a:a.shift(1).rolling(5,min_periods=3).mean()).reindex(ids).to_numpy()
 return {"flow_signed6":flow[ids],"thermal_flow_joint4":joint[ids],
  "meter_load6":load[ids],"flow_history8":np.column_stack([frame.iloc[ids].to_numpy(),hist])}
modelcode=oldcode[oldcode.index("def predict(parent,q):"):oldcode.index("records=[];choices=[]")]
modelcode=modelcode.replace("tr=views(z,ctx,ref);te=views(z,parent,q)","tr=views(z,ctx,ref,ref);te=views(z,parent,q,ref)")
exec(modelcode)
# The entire prediction/inner-selection loop is reused, with distinct output names.
runner=oldcode[oldcode.index("records=[];choices=[]"):oldcode.index("# Descriptive missed-case audit")]
runner=runner.replace("cn7_170_persistence","cn7_171_flow")
# Add new experts to the fixed 7/4 previous candidate, rather than replacing it.
needle="basepred=baseline.three_specialists_confirmed.to_numpy(int)"
replace="""basepred=baseline.three_specialists_confirmed.to_numpy(int)
previous=pd.read_csv(OUT/"cn7_170_persistence_predictions.csv",parse_dates=["TimeStamp"])
previous=previous[(previous.model=="persistent_abs4_ET")&(previous.rule=="normal95")].set_index("TimeStamp").loc[d.TimeStamp]
basepred=basepred|previous.prediction.to_numpy(int)"""
runner=runner.replace(needle,replace)
exec(runner)
base=stats(y[eligible],basepred[eligible])
primary=mm[mm.model.eq("inner_selected_mean3")]
top=mm.sort_values(["F1","FP"],ascending=[False,True]).head(12)
print("\nBASELINE 170",base)
print("\nPRIMARY\n",primary.to_string(index=False))
print("\nCANDIDATES\n",top.to_string(index=False))
bestrow=top.iloc[0]
bestpred=pp[(pp.model==bestrow.model)&(pp.rule==bestrow.rule)].set_index("TimeStamp").loc[d.TimeStamp]
pr=bestpred.prediction.to_numpy(int)
if bestrow.policy=="add_to_165":pr=pr|basepred
cases=d[["TimeStamp","gas","any_fail","run_id","seq"]].copy()
cases["previous_prediction"]=basepred
cases["candidate_prediction"]=pr
cases["new_alarm"]=(pr==1)&(basepred==0)
cases.to_csv(OUT/"cn7_171_flow_best_candidate_cases.csv",index=False,encoding="utf-8-sig")
print("\nNEW ALARMS\n",cases[cases.new_alarm].to_string(index=False))
print("\nMISSED\n",cases[cases.gas&cases.candidate_prediction.eq(0)].to_string(index=False))
prov={"source_input_sha256":hashlib.sha256(INPUT.read_bytes()).hexdigest(),
 "features":VIEW_NAMES,"validation":"same temporal20/purge10/nested5; not unseen-run or forward",
 "baseline":"165 confirmed OR 170 persistent_ET normal95",
 "caveat":"test-informed feature development; fixed best chosen after outer scores; primary inner selected also reported",
 "model_selection":"no outer labels used to choose per-fold cutoff; normal quantiles from inner predictions"}
(OUT/"cn7_171_flow_provenance.json").write_text(json.dumps(prov,ensure_ascii=False,indent=2),encoding="utf-8")
lines=["# CN7 가스 추가 개선: 상대적인 충전 상태와 동시 변화","",
 "정상592/가스13, 샷 기준. 170번의 7TP/4FP 후보를 고정하고 새로운 작은 전문가를 추가한다.",
 "같은 시간블록20/purge10 및 중첩검증5 유지. 평가 센서가 학습 이력에 들어가는 경우0.",
 "F1이 높은 고정 후보는 평가 결과를 보고 고른 사후 개발 최고이며 독립 성능이 아니다.",
 "비교표의 add_to_165는 코드 재사용 명칭으로 실제로는 165+170 기준에 추가한 것이다.","",
 "## 기준",str(base),"","## 안쪽 선택",primary.to_string(index=False),"",
 "## 고정 후보 탐색",top.to_string(index=False),"",
 "## 새 경보",cases[cases.new_alarm].to_string(index=False),"",
 "## 남은 미탐",cases[cases.gas&cases.candidate_prediction.eq(0)].to_string(index=False)]
(ROOT/"reports/cn7_171_flow_review.md").write_text("\n".join(lines),encoding="utf-8")


Finished 1/20: cn7_R1__B00


Finished 2/20: cn7_R1__B01


Finished 3/20: cn7_R1__B02


Finished 4/20: cn7_R1__B03


Finished 5/20: cn7_R1__B04


...

Finished 18/20: cn7_R2__B07


Finished 19/20: cn7_R2__B08


Finished 20/20: cn7_R2__B09



BASELINE 170 {'TP': 7, 'FP': 4, 'FN': 6, 'TN': 588, 'precision': 0.6363636363636364, 'recall': 0.5384615384615384, 'F1': 0.5833333333333334}

PRIMARY
                model            rule     policy   N  gas  TP  FP  FN  TN  precision   recall       F1  AP_standalone
inner_selected_mean3 inner_selection standalone 605   13   0  14  13 578       0.00 0.000000 0.000000       0.014933
inner_selected_mean3 inner_selection add_to_165 605   13   7  18   6 574       0.28 0.538462 0.368421       0.014933

CANDIDATES
                     model     rule     policy   N  gas  TP  FP  FN  TN  precision   recall       F1  AP_standalone
          flow_signed6_LR normal99 add_to_165 605   13   9   8   4 584   0.529412 0.692308 0.600000       0.086394
           meter_load6_ET normal99 add_to_165 605   13   7   6   6 586   0.538462 0.538462 0.538462       0.020270
        meter_load6_mean3 normal99 add_to_165 605   13   7   7   6 585   0.500000 0.538462 0.518519       0.013504
   thermal_flow_joint4_L

3494